### 1. import library

In [1]:
!pip install transformers sentencepiece protobuf Pillow pandas torch torchvision

In [2]:
import re
import random
import time
from statistics import mode

from PIL import Image
import numpy as np
import pandas
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from collections import Counter
from transformers import AutoTokenizer, AutoModel

torch.cuda.is_available()

/home/cm-pc/DLBasic2026_Final_Assignment_VQA/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### 2.utils

In [3]:
def set_seed(seed):
    """
    シードを固定する．

    Parameters
    ----------
    seed : int
        乱数生成に用いるシード値．
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
def process_text(text):
    """
    入力文と回答のフォーマットを統一するための関数．

    Parameters
    ----------
    text : str
        入力文，もしくは回答．
    """
    # lowercase
    text = text.lower()

    # 数詞を数字に変換
    num_word_to_digit = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
        'ten': '10'
    }
    for word, digit in num_word_to_digit.items():
        text = text.replace(word, digit)

    # 小数点のピリオドを削除
    text = re.sub(r'(?<!\d)\.(?!\d)', '', text)

    # 冠詞の削除
    text = re.sub(r'\b(a|an|the)\b', '', text)

    # 短縮形のカンマの追加
    contractions = {
        "dont": "don't", "isnt": "isn't", "arent": "aren't", "wont": "won't",
        "cant": "can't", "wouldnt": "wouldn't", "couldnt": "couldn't"
    }
    for contraction, correct in contractions.items():
        text = text.replace(contraction, correct)

    # 句読点をスペースに変換
    text = re.sub(r"[^\w\s':]", ' ', text)

    # 句読点をスペースに変換
    text = re.sub(r'\s+,', ',', text)

    # 連続するスペースを1つに変換
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [5]:
class VQADataset(torch.utils.data.Dataset):
    """
    VQA データセットを扱うためのクラス．
    """
    def __init__(self, df_path, image_dir, transform=None, answer=True):
        self.transform = transform  # 画像の前処理
        self.image_dir = image_dir  # 画像ファイルのディレクトリ
        self.df = pandas.read_json(df_path)  # 画像ファイルのパス，question, answerを持つDataFrame
        self.answer = answer

        # --- 変更点: AutoTokenizerの利用 ---
        self.text_model_name = 'microsoft/deberta-v3-base' # 'microsoft/deberta-v3-base' などに変更可能
        self.tokenizer = AutoTokenizer.from_pretrained(self.text_model_name)

        self.answer2idx = {}
        self.idx2answer = {}

        if self.answer:
            # 回答に含まれる文章を辞書に追加
            for answers in self.df["answers"]:
                for answer in answers:
                    word = answer["answer"]
                    word = process_text(word)
                    if word not in self.answer2idx:
                        self.answer2idx[word] = len(self.answer2idx)
            self.idx2answer = {v: k for k, v in self.answer2idx.items()}  # 逆変換用の辞書(answer)

    def update_dict(self, dataset):
        """
        検証用データ，テストデータの辞書を訓練データの辞書に更新する．
        """
        self.answer2idx = dataset.answer2idx
        self.idx2answer = dataset.idx2answer

    def __getitem__(self, idx):
        """
        対応するidxのデータ（画像，質問，回答）を取得．
        """
        image = Image.open(f"{self.image_dir}/{self.df['image'][idx]}")
        image = self.transform(image)

        question_text = self.df["question"][idx]

        encoded = self.tokenizer(
            question_text,
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors='pt'
        )

        inputs_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)

        if self.answer:
            answers = [self.answer2idx[process_text(answer["answer"])] for answer in self.df["answers"][idx]]

            soft_label = torch.zeros(len(self.answer2idx))
            answer_counts = Counter(answers)

            for ans_id, count in answer_counts.items():
                soft_label[ans_id] = min(count / 3.0, 1.0)

            return image, inputs_ids, attention_mask, torch.Tensor(answers), soft_label
        else:
            return image, inputs_ids, attention_mask

    def __len__(self):
        return len(self.df)

In [6]:
def VQA_criterion(batch_pred, batch_answers):
    """
    VQA タスクに用いられる評価関数．
    """
    total_acc = 0.

    for pred, answers in zip(batch_pred, batch_answers):
        acc = 0.
        for i in range(len(answers)):
            num_match = 0
            for j in range(len(answers)):
                if i == j:
                    continue
                if pred == answers[j]:
                    num_match += 1
            acc += min(num_match / 3, 1)
        total_acc += acc / 10

    return total_acc / len(batch_pred)

### 3.model

In [7]:
class VQAModel(nn.Module):
    """
    VQA タスクを解くためのモデル例．
    """
    def __init__(self, n_answer: int):
        """
        コンストラクタ．

        Parameters
        ----------
        n_answer: int
            出力のクラス数
        """
        super().__init__()
        # 1. 事前学習済みVision Transformer (ViT-B/16) のロード
        self.vit = torchvision.models.vit_b_16(weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
        # 分類ヘッドを無効化して特徴量抽出器として利用 (出力次元: 768)
        self.vit.heads = nn.Identity()

        # テキストエンコーダを元の microsoft/deberta-v3-base に戻す
        text_model_name = 'microsoft/deberta-v3-base'
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.text_encoder.gradient_checkpointing_enable()

        # Cross-Attention: embed_dim をテキストの hidden_size に合わせる
        self.cross_attn = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        # Attention を通した特徴量と画像特徴量（射影後）を結合して出力層へ
        self.fc = nn.Sequential(
            nn.Linear(768 * 2, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, n_answer)
        )

    def forward(self, image, input_ids, attention_mask):
        # 画像の特徴量を取得: [batch_size, image_feat_dim]
        image_feature = self.vit(image).to(torch.float32)

        # テキストの系列全体の特徴量を取得: [batch_size, seq_len, text_hidden]
        self.text_encoder = self.text_encoder.float()
        with torch.amp.autocast('cuda', enabled=False):
            text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            text_seq = text_outputs.last_hidden_state.to(torch.float32)

        # --- Cross-Attentionの適用 ---
        # 画像特徴量をQueryとして [batch_size, 1, text_hidden] の形状にする
        query = image_feature.unsqueeze(1)
        # テキスト特徴量をKey, Valueとする
        key = text_seq
        value = text_seq

        # BERTのpadding部分をAttentionから除外するマスクを作成
        key_padding_mask = (attention_mask == 0)
        key_padding_mask[:, 0] = False # 先頭の[CLS]トークンだけは絶対にマスクしない

        # Attentionの計算: attn_outputは [batch_size, 1, text_hidden]
        attn_output, _ = self.cross_attn(query, key, value, key_padding_mask=key_padding_mask)
        attn_output = attn_output.squeeze(1) # [batch_size, text_hidden] に戻す

        # 元の画像特徴量（射影後）とAttentionの出力を結合して予測
        x = torch.cat([image_feature, attn_output], dim=1)
        x = self.fc(x)

        return x

### 4.train

In [8]:
def train(model, dataloader, optimizer, criterion, device):
    """
    学習用の関数．
    """
    model.train()

    total_loss = 0
    total_acc = 0
    simple_acc = 0

    start = time.time()
    for image, input_id, attention_mask, answers, soft_label in dataloader:
        image, input_id, attention_mask, answers, soft_label = \
            image.to(device), input_id.to(device), attention_mask.to(device), answers.to(device), soft_label.to(device)

        pred = model(image, input_id, attention_mask)
        loss = criterion(pred, soft_label)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)  # VQA accuracy
        simple_acc += (pred.argmax(1) == soft_label.argmax(1)).float().mean().item()  # simple accuracy

    return total_loss / len(dataloader), total_acc / len(dataloader), simple_acc / len(dataloader), time.time() - start



def eval(model, dataloader, optimizer, criterion, device):
    """
    評価用の関数．
    """
    model.eval()

    total_loss = 0
    total_acc = 0
    simple_acc = 0

    start = time.time()
    for image, input_id, attention_mask, answers, soft_label in dataloader:
        image, input_id, attention_mask, answers, soft_label = \
            image.to(device), input_id.to(device), attention_mask.to(device), answers.to(device), soft_label.to(device)

        pred = model(image, input_id, attention_mask)
        loss = criterion(pred, soft_label)

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)  # VQA accuracy
        simple_acc += (pred.argmax(1) == soft_label.argmax(1)).float().mean().item()  # simple accuracy

    return total_loss / len(dataloader), total_acc / len(dataloader), simple_acc / len(dataloader), time.time() - start

### 5. make submission file

In [9]:
# deviceの設定
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. VQA向けの安全なData Augmentation
# 反転や大きな回転は避け、軽微なクロップと色調変化に留める
imagenet_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # ズームインによる位置ズレ
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.0), # 軽微な色変化
    transforms.ToTensor(),
    imagenet_norm
])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    imagenet_norm
])

from pathlib import Path

data_dir = Path("/home/cm-pc/DLBasic2026_Final_Assignment_VQA/data")

# train_datasetにtransform_trainを適用
train_dataset = VQADataset(df_path=data_dir / "train.json", image_dir=data_dir / "train", transform=transform_train)
test_dataset = VQADataset(df_path=data_dir / "valid.json", image_dir=data_dir / "valid", transform=transform, answer=False)
test_dataset.update_dict(train_dataset)

batch_size = 4
grad_accum_steps = 8  # 実効バッチサイズ = 4 * 8 = 32

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

model = VQAModel(n_answer=len(train_dataset.answer2idx)).to(device)
scaler = torch.amp.GradScaler('cuda') if device == "cuda" else None

if device == "cuda":
    torch.cuda.empty_cache()

# optimizer / criterion
num_epoch = 15
criterion = nn.BCEWithLogitsLoss()

# 3. 層ごとの学習率（Parameter Groups）の設定
# 画像・テキストの事前学習モデルは低い学習率、融合層は高めに設定する
optimizer = torch.optim.AdamW([
    {'params': model.vit.parameters(), 'lr': 5e-6},
    {'params': model.text_encoder.parameters(), 'lr': 5e-6},
    {'params': model.cross_attn.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-4}
], weight_decay=1e-4)

/home/cm-pc/DLBasic2026_Final_Assignment_VQA/lib/python3.8/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/home/cm-pc/DLBasic2026_Final_Assignment_VQA/lib/python3.8/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte

In [11]:
# from pathlib import Path
# import re
#
# checkpoint_dir = Path("/home/cm-pc/DLBasic2026_Final_Assignment_VQA/models/model1/checkpoints")
# checkpoint_dir.mkdir(parents=True, exist_ok=True)
#
# checkpoint_pattern = re.compile(r"epoch_(\d+)\.pt$")
# checkpoint_files = []
# for checkpoint_path in checkpoint_dir.glob("epoch_*.pt"):
#     match = checkpoint_pattern.search(checkpoint_path.name)
#     if match is not None:
#         checkpoint_files.append((int(match.group(1)), checkpoint_path))
#
# start_epoch = 0
# if checkpoint_files:
#     latest_epoch, latest_checkpoint_path = max(checkpoint_files, key=lambda item: item[0])
#     checkpoint = torch.load(latest_checkpoint_path, map_location=device)
#
#     if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
#         model.load_state_dict(checkpoint["model_state_dict"])
#         if "optimizer_state_dict" in checkpoint:
#             optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
#         start_epoch = int(checkpoint.get("epoch", latest_epoch))
#     else:
#         model.load_state_dict(checkpoint)
#         start_epoch = latest_epoch
#
#     print(f"Loaded checkpoint from {latest_checkpoint_path} (resume from epoch {start_epoch + 1})")
# else:
#     print("No checkpoint found. Start training from scratch.")

for epoch in range(num_epoch):
    train_loss, train_acc, train_simple_acc, train_time = train(
        model, train_loader, optimizer, criterion, device
    )
    print(f"【{epoch + 1}/{num_epoch}】\n"
            f"train time: {train_time:.2f} [s]\n"
            f"train loss: {train_loss:.8f}\n"
            f"train acc: {train_acc:.4f}\n"
            f"train simple acc: {train_simple_acc:.4f}")

    # checkpoint_path = checkpoint_dir / f"epoch_{epoch + 1:02d}.pt"
    # torch.save(
    #     {
    #         "epoch": epoch + 1,
    #         "model_state_dict": model.state_dict(),
    #         "optimizer_state_dict": optimizer.state_dict(),
    #         "train_loss": train_loss,
    #         "train_acc": train_acc,
    #         "train_simple_acc": train_simple_acc,
    #     },
    #     checkpoint_path,
    # )
    # print(f"Saved checkpoint to {checkpoint_path}")


/home/cm-pc/DLBasic2026_Final_Assignment_VQA/lib/python3.8/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


【1/15】
train time: 916.91 [s]
train loss: 0.00358309
train acc: 0.3984
train simple acc: 0.3623
【2/15】
train time: 917.07 [s]
train loss: 0.00047792
train acc: 0.4869
train simple acc: 0.4419
【3/15】
train time: 917.26 [s]
train loss: 0.00042469
train acc: 0.5336
train simple acc: 0.4815
【4/15】
train time: 902.26 [s]
train loss: 0.00037512
train acc: 0.5599
train simple acc: 0.5063
【5/15】
train time: 916.73 [s]
train loss: 0.00033860
train acc: 0.5816
train simple acc: 0.5267
【6/15】
train time: 889.87 [s]
train loss: 0.00031204
train acc: 0.6067
train simple acc: 0.5497
【7/15】
train time: 887.86 [s]
train loss: 0.00029140
train acc: 0.6266
train simple acc: 0.5669
【8/15】
train time: 887.63 [s]
train loss: 0.00027565
train acc: 0.6455
train simple acc: 0.5834
【9/15】
train time: 888.06 [s]
train loss: 0.00026207
train acc: 0.6654
train simple acc: 0.6018
【10/15】
train time: 888.22 [s]
train loss: 0.00025114
train acc: 0.6783
train simple acc: 0.6152
【11/15】
train time: 888.51 [s]
train lo

In [12]:
import os
import glob
import torch
import numpy as np

# deviceの定義を追加
device = "cuda" if torch.cuda.is_available() else "cpu"

model_dir = Path("/home/cm-pc/DLBasic2026_Final_Assignment_VQA/models/model1")
model_path = os.path.join(model_dir, "model.pt")


# モデルが未定義の場合は初期化する
if 'model' not in globals() and 'model' not in locals():
    model = VQAModel(n_answer=len(train_dataset.answer2idx)).to(device)

# 1. 最新のチェックポイントから重みを読み込む
# checkpoints = glob.glob(os.path.join(checkpoint_dir, "epoch_*.pt"))
# if checkpoints:
#     # ファイル名の 'epoch_XX' の数字部分を取得して正しくソートし、最新を取得
#     latest_checkpoint = sorted(checkpoints, key=lambda x: int(os.path.splitext(os.path.basename(x))[0].split('_')[1]))[-1]
#     print(f"{latest_checkpoint} から重みを読み込みます...")
#     checkpoint = torch.load(latest_checkpoint, map_location=device)
#     if "model_state_dict" in checkpoint:
#         model.load_state_dict(checkpoint["model_state_dict"])
#     else:
#         model.load_state_dict(checkpoint)
# else:
#     print("チェックポイントファイルが見つかりません。現在のメモリ上のモデルを使用します。")

model = model.to(device)
model.eval()

# 2. 推論を行って submission ファイルを作成
submission = []
print("推論を開始します...")
with torch.no_grad():  # 推論時は勾配計算を無効化してメモリ節約・高速化
    for image, input_ids, attention_mask in test_loader:
        image, input_ids, attention_mask = image.to(device), input_ids.to(device), attention_mask.to(device)
        pred = model(image, input_ids, attention_mask)
        pred = pred.argmax(1).cpu().item()
        submission.append(pred)

submission = [train_dataset.idx2answer[id] for id in submission]
submission = np.array(submission)

# モデルの重みを FP16 にキャストしてサイズを削減 (約1.1GB -> 約550MB)
state_dict_fp16 = {k: v.cpu().half() for k, v in model.state_dict().items()}

torch.save(state_dict_fp16, "model.pt")
os.makedirs(model_dir, exist_ok=True)
torch.save(state_dict_fp16, model_path)
np.save("submission.npy", submission)
print("submission.npy と model.pt (FP16化済) の作成・保存が完了しました。")


推論を開始します...
submission.npy と model.pt (FP16化済) の作成・保存が完了しました。


In [14]:
from zipfile import ZipFile

zip_file_path = os.path.join(model_dir, "submission.zip")
model_path = "model.pt"
notebook_path = "/home/cm-pc/DLBasic2026_Final_Assignment_VQA/src/train.ipynb"

with ZipFile(zip_file_path, "w") as zf:
    zf.write("submission.npy")
    zf.write(model_path)
    zf.write(notebook_path, arcname="train.ipynb")